# 🤖 REFLECT — Mood Classifier Training (Google Colab)

**Goal:** Train a text classification model that predicts mood from journal text.

**Labels:** `happy`, `calm`, `neutral`, `sad`, `anxious`

**Output:** `mood_classifier.tflite` — drop this file into `app/src/main/assets/`

---
### Steps:
1. Run all cells top to bottom
2. Download `mood_classifier.tflite` at the end
3. Copy it to `app/src/main/assets/mood_classifier.tflite` in Android Studio
4. Rebuild the app — AI mode activates automatically


In [ ]:
# Cell 1: Install dependencies
!pip install tensorflow numpy scikit-learn -q

In [ ]:
# Cell 2: Training dataset
# Each entry: (text, label)
# Labels: 0=happy, 1=calm, 2=neutral, 3=sad, 4=anxious

import numpy as np

training_data = [
    # HAPPY (0)
    ("I feel so happy and excited today!", 0),
    ("Had an amazing day with my friends, everything went great", 0),
    ("Achieved my goal today, feeling proud and grateful", 0),
    ("So thankful for all the wonderful things in my life", 0),
    ("Celebrated my success, feeling joyful and blessed", 0),
    ("Everything is going fantastic, love this positive energy", 0),
    ("Today was perfect, laughed so much and enjoyed every moment", 0),
    ("Feeling glad and thrilled about the new opportunity", 0),
    ("Best day ever, my hard work finally paid off", 0),
    ("I am delighted and excited about what lies ahead", 0),
    ("Great progress on my goals today, feeling wonderful", 0),
    ("Completed everything on my list, so proud of myself", 0),
    ("Had fun and enjoyed a brilliant day full of laughter", 0),
    ("Feeling positive and bright, life is excellent right now", 0),
    ("Received good news today, feeling truly blessed and happy", 0),

    # CALM (1)
    ("Feeling peaceful and relaxed after my meditation session", 1),
    ("Had a quiet serene morning, felt very calm and present", 1),
    ("Everything feels balanced and steady today", 1),
    ("Took a gentle walk in nature, feeling tranquil", 1),
    ("Breathed deeply and found my center today", 1),
    ("Mindful and content, enjoying the still moments", 1),
    ("Comfortable and at ease, nothing bothers me today", 1),
    ("Slow quiet day, feeling very relaxed and comfortable", 1),
    ("Clarity of mind today, serene and flowing", 1),
    ("Centered and present, feeling the gentle calm around me", 1),
    ("Peaceful evening reflecting on the good things in my life", 1),
    ("Rested well and woke up with a tranquil mind", 1),
    ("Went for a slow morning walk, feeling steady and mindful", 1),
    ("Meditating helped me find balance and inner peace", 1),
    ("Just feeling easy and content, nothing special happened", 1),

    # NEUTRAL (2)
    ("Nothing special happened today, just a normal day", 2),
    ("It was an average day, went to work came home", 2),
    ("Mixed feelings today, not good not bad", 2),
    ("Okay day, nothing to complain about but nothing exciting", 2),
    ("Just regular daily routine, both positive and negative moments", 2),
    ("Standard week, moderate progress on everything", 2),
    ("Fine day overall, things were just okay today", 2),
    ("Alright, nothing unusual happened, usual stuff", 2),
    ("Kind of unsure how I feel today, mixed emotions", 2),
    ("Regular morning routine, nothing out of the ordinary", 2),
    ("Just an ordinary day, felt somewhat fine throughout", 2),
    ("Normal progress today, nothing too bad nothing too great", 2),
    ("Going through the motions, felt balanced neither happy nor sad", 2),
    ("Today was standard, moderate energy throughout the day", 2),
    ("Both good and not so good things happened today", 2),

    # SAD (3)
    ("Feeling really sad and lonely today", 3),
    ("I cried a lot, everything feels broken and hopeless", 3),
    ("Miss someone so much, feeling empty and lost", 3),
    ("Disappointed with myself, feeling down and depressed", 3),
    ("Hurt by what happened, grief is overwhelming me", 3),
    ("Feeling gloomy and dark, nothing seems to go right", 3),
    ("Tears won't stop, feeling heartbroken and miserable", 3),
    ("Low energy and sad mood all day long", 3),
    ("Lost my motivation, feel like giving up on everything", 3),
    ("Sorrow fills my heart, feeling completely alone", 3),
    ("Crying because I failed, feeling so upset and blue", 3),
    ("Nothing feels good today, deeply unhappy and disappointed", 3),
    ("Missed an important opportunity, feeling really bad", 3),
    ("Dark thoughts today, feeling worthless and hopeless", 3),
    ("Feeling broken inside, pain is too much to handle", 3),

    # ANXIOUS (4)
    ("So anxious and worried about the upcoming deadline", 4),
    ("Feeling nervous and stressed about everything at work", 4),
    ("Overwhelmed with too much on my plate, can't breathe", 4),
    ("Scared about the future, fear is taking over", 4),
    ("Panic attack today, feeling tense and out of control", 4),
    ("Restless and uneasy, dread keeps creeping in", 4),
    ("Too many things to do, pressure is unbearable", 4),
    ("Rushing through everything, very stressed and nervous", 4),
    ("Exhausted and anxious, can't focus on anything properly", 4),
    ("Afraid of failing, struggling with everything right now", 4),
    ("Jittery and worried about my presentation tomorrow", 4),
    ("Insecure and uncertain, freaking out about changes", 4),
    ("Sleepless night due to worry, feeling exhausted and tense", 4),
    ("Work stress is overwhelming, cannot stop worrying", 4),
    ("Feeling panicky and anxious, difficult to stay calm", 4),
]

texts  = [d[0] for d in training_data]
labels = [d[1] for d in training_data]
print(f'Dataset size: {len(texts)} samples')
print(f'Label distribution: happy={labels.count(0)}, calm={labels.count(1)}, neutral={labels.count(2)}, sad={labels.count(3)}, anxious={labels.count(4)}')

In [ ]:
# Cell 3: Tokenize and build vocabulary

import re
from collections import Counter

VOCAB_SIZE = 200
MAX_SEQ_LEN = 50

def tokenize(text):
    return re.sub(r'[^a-z ]', ' ', text.lower()).split()

# Build vocab from training data
all_words = []
for t in texts:
    all_words.extend(tokenize(t))

word_counts = Counter(all_words)
vocab_words = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word_to_id = {w: i for i, w in enumerate(vocab_words)}

def encode(text):
    tokens = tokenize(text)
    ids = [word_to_id.get(t, 1) for t in tokens]  # 1 = <UNK>
    # Pad or truncate to MAX_SEQ_LEN
    ids = ids[:MAX_SEQ_LEN] + [0] * max(0, MAX_SEQ_LEN - len(ids))
    return ids

X = np.array([encode(t) for t in texts], dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'Vocabulary size: {len(vocab_words)}')

In [ ]:
# Cell 4: Build and train the model

import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAX_SEQ_LEN,)),
    tf.keras.layers.Embedding(input_dim=len(vocab_words), output_dim=16, input_length=MAX_SEQ_LEN),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(5, activation='softmax')  # 5 mood classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=8,
    validation_data=(X_val, y_val),
    verbose=1
)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f'\n✅ Validation accuracy: {val_acc:.2%}')

In [ ]:
# Cell 5: Test the model with sample sentences

LABEL_NAMES = ['happy', 'calm', 'neutral', 'sad', 'anxious']
EMOJIS      = ['😄', '😌', '😐', '😔', '😰']

test_sentences = [
    "I feel so happy and excited today!",
    "Peaceful morning meditation, feeling calm and centered",
    "Nothing special today, just a regular day",
    "Feeling really sad, miss my family so much",
    "Stressed out about the deadline, so many things to do",
    "Had a great week, achieved all my goals and feel grateful",
    "Quiet evening, reading a book feeling relaxed",
    "Crying a lot today, feeling broken and lost",
    "Worried about failing my exam tomorrow",
    "Today was okay, not good not bad, just normal"
]

print('\n📊 Model Predictions:\n')
for sentence in test_sentences:
    encoded = np.array([encode(sentence)], dtype=np.float32)
    scores  = model.predict(encoded, verbose=0)[0]
    pred_id = np.argmax(scores)
    print(f'{EMOJIS[pred_id]} {LABEL_NAMES[pred_id]:8} ({scores[pred_id]:.0%}) — "{sentence[:50]}"')

In [ ]:
# Cell 6: Convert to TFLite and save vocab

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization for smaller size
tflite_model = converter.convert()

# Save model
with open('mood_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

# Save vocabulary (MUST match Android MoodClassifier.java)
with open('mood_vocab.txt', 'w') as f:
    for word in vocab_words:
        f.write(word + '\n')

model_size = len(tflite_model) / 1024
print(f'✅ mood_classifier.tflite saved  ({model_size:.1f} KB)')
print(f'✅ mood_vocab.txt saved  ({len(vocab_words)} words)')
print()
print('📋 NEXT STEPS:')
print('  1. Download mood_classifier.tflite (Files panel on left)')
print('  2. Download mood_vocab.txt')
print('  3. Copy BOTH to:  app/src/main/assets/')
print('  4. Rebuild the app in Android Studio')
print('  5. The AI button on Add Reflection screen will now use the trained model!')

In [ ]:
# Cell 7: Download files directly from Colab
from google.colab import files
files.download('mood_classifier.tflite')
files.download('mood_vocab.txt')

In [ ]:
# Cell 8 (OPTIONAL): Plot training accuracy graph
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],   label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_graph.png', dpi=120)
plt.show()
print('✅ Graph saved as training_graph.png')